# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Matías Taborda
- Alumno 2 :    -

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.datasets import fetch_lfw_people
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Dispositivo: {device}")

PyTorch: 2.11.0+cpu | Dispositivo: cpu


In [ ]:
# Cargo el dataset LFW como base
lfw = fetch_lfw_people(min_faces_per_person=70, color=True, resize=1.0)

X_lfw = lfw.images    # (n_samples, 125, 94, 3) float en [0, 1]
y_lfw = lfw.target     # etiquetas numéricas
NOMBRES_LFW = lfw.target_names

print(f"Total de imágenes: {X_lfw.shape[0]}")
print(f"Tamaño de imagen: {X_lfw.shape[1:3]}")
print(f"Rango de valores: [{X_lfw.min():.2f}, {X_lfw.max():.2f}]")
print(f"\nPersonas ({len(NOMBRES_LFW)}):")

from collections import Counter
conteo = Counter(y_lfw)
for idx, nombre in enumerate(NOMBRES_LFW):
    print(f"  {nombre}: {conteo[idx]} imágenes")

Total de imágenes: 1288
Tamaño de imagen: (125, 94)
Rango de valores: [0.00, 1.00]

Personas (7):
  Ariel Sharon: 77 imágenes
  Colin Powell: 236 imágenes
  Donald Rumsfeld: 121 imágenes
  George W Bush: 530 imágenes
  Gerhard Schroeder: 109 imágenes
  Hugo Chavez: 71 imágenes
  Tony Blair: 144 imágenes


In [52]:
# Verifico mi dataset 

from pathlib import Path
dataset_path = Path("src/data/dataset_tp1")
dataset_folders = []
 
for folder in dataset_path.iterdir():
    if folder.is_dir():
        dataset_folders.append(folder)

# verifico imágenes en las carpetas
print("*"*20)
print("Dataset TP1")
print("*"*20)
for folder in dataset_folders:
    print(f"{folder.name}: {len(list(folder.iterdir()))} imágenes")

********************
Dataset TP1
********************
person_1: 1 imágenes
person_2: 0 imágenes
person_3: 0 imágenes


In [55]:
# Recorro las carpetas y cargo las imágenes 
X = []
y = []
nombres = []
for persona_id, folder in enumerate(dataset_path.iterdir()):
    if folder.is_dir():
        nombres.append(folder.name)
        for img_file in folder.glob('*.jpg'):
            img = Image.open(img_file).convert('RGB')
            img = img.resize((94, 125))
            img_array = np.array(img) / 255.0
            X.append(img_array) 
            y.append(persona_id) 
 
X = np.array(X)
y = np.array(y)

print(f"Dataset: {X.shape[0]} imágenes, {len(nombres)} personas")

Dataset: 1 imágenes, 3 personas
